<a href="https://colab.research.google.com/github/siva123995/ProjectLLM/blob/main/MultiAgent-CareerAgents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -qU langchain langchain_community langgraph tavily-python langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.8 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('Secret')
GOOGLE_API_KEY=userdata.get('sivathede')
TAVILY_API_KEY = userdata.get('TavilyKey')

In [14]:
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from tavily import TavilyClient
from typing import List
from langchain_community.tools import tool
from langgraph.graph import StateGraph, START, END
tavily = TavilyClient(api_key=TAVILY_API_KEY)

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="models/gemini-3.5-flash", google_api_key=GOOGLE_API_KEY)

In [47]:
@tool
def parse_from_resume(text:str) -> str:
  "Extract skills, technologies, years of experience from resume"
  prompt = f"Extract the relevant skills, years of experience, and location from the resume:{text}"
  prompt = f"Extract structured info like skills, roles, and years of experience and other relevant information from:{text}"
  response = model.invoke(text).content
  return response

@tool
def search_jobs_tavily(query: str) -> str:
  "searches jobs using tavily in the internet"
  results = tavily.search(query)
  # returning Query and followUp questions and results, So, let's pick only results
  print(f"---> {results=}")
  return results["results"]

def profile(state: List[BaseMessage]) -> List[BaseMessage]:
  resume = state[-1].content
  profile = parse_from_resume.invoke(resume)
  print(f'--->{profile=}')
  return state + [AIMessage(content = f"Extracted profile: \n {profile} ")]

def job_search_tool(state:List[BaseMessage]) -> List[BaseMessage]:
  profile = state[-1].content
  # remember model.invoke always return in below format
  # AIMessage[{
  #     'type':'text',
  #     'text':'<content>'
  #    }]
  query_message = model.invoke(f"Based on the profile, write a job search query: {profile} keep the search engine very short so search engines to search")
  # you need this line, because when you pass query to tavily, it should be string or dictionary, So, parsing them properly.
  query = query_message.content[0]['text'] if isinstance(query_message.content, list) else query_message.content
  print(f'====>{query=}')
  jobs = search_jobs_tavily.invoke(query)
  print(f'---> {jobs=}')
  return state + [AIMessage(content=f"Found jobs are: {jobs}")]

def matcher_node(state: List[BaseMessage]) -> List[BaseMessage]:
  jobs = state[-1].content
  profile = state[-2].content
  prompt = f"Based on the given profile {profile} and Recommend top 3 jobs from the jobs {jobs}"
  top_jobs = model.invoke([HumanMessage(content=prompt)])
  # remember model.invoke always return in below format
  # AIMessage[{
  #     'type':'text',
  #     'text':'<content>'
  #    }]
  return state + [top_jobs]

def cover_letter(state: List[BaseMessage]) -> List[BaseMessage]:
  jobs = state[-1].content
  profile = state[-3].content
  prompt = f"Write a personalised short cover letter on the job: {jobs} based on the profile: {profile}"
  letter = model.invoke([HumanMessage(content=prompt)])
  print(f'==>{letter=}')
  return state + [letter]

graph = StateGraph(List[BaseMessage])

graph.add_node("profile", profile)
graph.add_node("search", job_search_tool)
graph.add_node("match", matcher_node)
graph.add_node("cover_letter", cover_letter)


graph.add_edge(START, "profile")
graph.add_edge("profile", "search")
graph.add_edge("search", "match")
graph.add_edge("match", "cover_letter")
graph.add_edge("cover_letter", END)

agent = graph.compile()

input_msg = [HumanMessage(content="I'm a senior Data engineer with 15 years of experience in Python, Pyspark, Spark, DataBricks, Azure, Based in Queensland Australia")]
response = agent.invoke(input_msg)

--->profile=[{'type': 'text', 'text': "G'day! That is an incredibly strong profile. With 15 years in the data space, you’ve likely seen the entire evolution from on-prem Hadoop/MapReduce and early Spark up to the modern Lakehouse era with Databricks and Azure. \n\nThe **Azure + Databricks + PySpark** stack is arguably the most dominant enterprise data architecture in Australia right now, and Queensland (particularly Brisbane, but also the resources/energy sector across the state) has a massive appetite for your skillset.\n\nSince you are at a Principal/Lead/Senior Architect level, I’d love to know how I can best support you. Here are a few ways we can collaborate:\n\n### 1. Architectural & Deep Technical Brainstorming\n*   **Databricks Optimization:** Deep-diving into Catalyst Optimizer issues, AQE tuning, custom partition strategies, or optimizing Photon engine workloads.\n*   **Unity Catalog & Governance:** Best practices for implementing Unity Catalog across multi-workspace Azure en

In [48]:
for msg in response:
  print(f"{type(msg).__name__}: {msg.content}\n")

HumanMessage: I'm a senior Data engineer with 15 years of experience in Python, Pyspark, Spark, DataBricks, Azure, Based in Queensland Australia

AIMessage: Extracted profile: 
 [{'type': 'text', 'text': "G'day! That is an incredibly strong profile. With 15 years in the data space, you’ve likely seen the entire evolution from on-prem Hadoop/MapReduce and early Spark up to the modern Lakehouse era with Databricks and Azure. \n\nThe **Azure + Databricks + PySpark** stack is arguably the most dominant enterprise data architecture in Australia right now, and Queensland (particularly Brisbane, but also the resources/energy sector across the state) has a massive appetite for your skillset.\n\nSince you are at a Principal/Lead/Senior Architect level, I’d love to know how I can best support you. Here are a few ways we can collaborate:\n\n### 1. Architectural & Deep Technical Brainstorming\n*   **Databricks Optimization:** Deep-diving into Catalyst Optimizer issues, AQE tuning, custom partition

In [20]:
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)

print('Available models:')
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-previ

Please run the above cell. Once executed, I will analyze the output and suggest an appropriate model to replace `gemini-pro`.